In [29]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

ROOT_DIR = Path.cwd().parent

sys.path.insert(0, str(ROOT_DIR))

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODEL_DIR = ROOT_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", ROOT_DIR)
print("Processed:", PROCESSED_DIR)
print("Models:", MODEL_DIR)

Project: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade
Processed: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\data\processed
Models: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\models


In [30]:
df = pd.read_csv(
    PROCESSED_DIR / "AAPL_features.csv"
)

print(df.columns.tolist())

['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'SMA_10', 'SMA_20', 'SMA_50', 'EMA_20', 'Momentum_5', 'Momentum_10', 'Momentum_20', 'RSI', 'MACD', 'MACD_Signal', 'BB_High', 'BB_Low', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'Return_Lag_5', 'Return_Lag_10', 'Target']


In [31]:
FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return",
    "SMA_10",
    "SMA_20",
    "SMA_50",
    "EMA_20",
    "Momentum_5",
    "Momentum_10",
    "Momentum_20",
    "RSI",
    "MACD",
    "MACD_Signal",
    "BB_High",
    "BB_Low",
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_3",
    "Return_Lag_5",
    "Return_Lag_10"
]

TARGET = "Target"

print("Number of features:", len(FEATURE_COLUMNS))

Number of features: 23


In [32]:
def train_stock_model(ticker):

    print("\n" + "=" * 60)
    print(f"Training model for {ticker}")
    print("=" * 60)

    file_path = (
        PROCESSED_DIR /
        f"{ticker}_features.csv"
    )

    if not file_path.exists():
        print(f"❌ File not found: {file_path}")
        return None

    df = pd.read_csv(file_path)

    # Keep only required columns
    required_columns = (
        FEATURE_COLUMNS + [TARGET]
    )

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        print(
            "❌ Missing columns:",
            missing_columns
        )
        return None

    df = df.dropna(
        subset=required_columns
    ).reset_index(drop=True)

    X = df[FEATURE_COLUMNS]
    y = df[TARGET]

    # Time-series split
    split_index = int(
        len(df) * 0.80
    )

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]

    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    print("Total:", len(df))
    print("Train:", len(X_train))
    print("Test :", len(X_test))

    # Model
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    # Prediction
    y_pred = model.predict(
        X_test
    )

    # Metrics
    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    # Save model
    model_path = (
        MODEL_DIR /
        f"{ticker}_model.pkl"
    )

    feature_path = (
        MODEL_DIR /
        f"{ticker}_feature_columns.pkl"
    )

    joblib.dump(
        model,
        model_path
    )

    joblib.dump(
        FEATURE_COLUMNS,
        feature_path
    )

    print(
        f"✅ Model saved: {model_path}"
    )

    return {
        "Ticker": ticker,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [33]:
TICKERS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "TSLA",
    "NVDA"
]

results = []

for ticker in TICKERS:

    result = train_stock_model(
        ticker
    )

    if result is not None:
        results.append(result)


Training model for AAPL
Total: 2871
Train: 2296
Test : 575
MAE  : 47.4308
RMSE : 57.9488
R²   : -1.8214
✅ Model saved: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\models\AAPL_model.pkl

Training model for MSFT
Total: 2871
Train: 2296
Test : 575
MAE  : 31.7685
RMSE : 45.7659
R²   : -0.0387
✅ Model saved: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\models\MSFT_model.pkl

Training model for GOOGL
Total: 2871
Train: 2296
Test : 575
MAE  : 73.0473
RMSE : 104.9582
R²   : -0.8921
✅ Model saved: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\models\GOOGL_model.pkl

Training model for AMZN
Total: 2871
Train: 2296
Test : 575
MAE  : 32.6060
RMSE : 40.3402
R²   : -1.4645
✅ Model saved: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\models\AMZN_model.pkl

Training model for TSLA
Total: 2871
Train: 2296
Test : 575
MAE  : 25.9017
RMSE : 37.0144
R²   : 0.8155
✅ Model saved: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade\models\TSLA_model.pkl

Training model for NVDA
Total: 2871
Train: 2296
Test : 575
MAE

In [34]:
results_df = pd.DataFrame(
    results
)

results_df

,Ticker,MAE,RMSE,R2
0,AAPL,47.430827,57.948750,-1.821385
1,MSFT,31.768462,45.765902,-0.038691
2,GOOGL,73.047287,104.958164,-0.892081
3,AMZN,32.605997,40.340182,-1.464475
4,TSLA,25.901683,37.014415,0.815483
5,NVDA,67.039946,76.008055,-3.446129


In [35]:
results_df.to_csv(
    PROCESSED_DIR / "model_performance.csv",
    index=False
)

print(
    "✅ Model performance saved."
)

✅ Model performance saved.


In [36]:
print("Saved models:")

for file in MODEL_DIR.glob("*.pkl"):
    print(file.name)

Saved models:
AAPL_feature_columns.pkl
AAPL_model.pkl
AMZN_feature_columns.pkl
AMZN_model.pkl
best_model.pkl
feature_columns.pkl
GOOGL_feature_columns.pkl
GOOGL_model.pkl
MSFT_feature_columns.pkl
MSFT_model.pkl
NVDA_feature_columns.pkl
NVDA_model.pkl
TSLA_feature_columns.pkl
TSLA_model.pkl
